# DIAGNOSTIC TOOL

In [1]:
import numpy as np
import pandas as pd
import skimage as ski
import matplotlib.pyplot as plt
import tifffile
import zarr
import os
import xml.etree.ElementTree as ET
import napari
from importlib import reload

In [2]:
import load_assess_image as load_assess

In [3]:
import qc_widget


In [11]:
main_dir=rf'D:\thu\HTAN\images\ILC\level_2'
file_name='TNP-TMA-29_A7-ILC-3A.ome.tif'
image_test=load_assess.AssessImage(main_dir=main_dir,
                    filename=file_name)

In [12]:
z = zarr.open(image_test.loader.image_store, mode='r')
z['0'].shape   # (n_channels, height, width) at resolution level 0


(35, 5201, 5051)

# QC Stats

In [13]:
qc_df=image_test.get_stats(resolution_level=2)

In [22]:
biomarker=list(image_test.biomarker_channel)

In [23]:
image_test.view_images(biomarkers_of_interest=biomarker,resolution_level=0)

Viewer(mouse_move_callbacks=[], mouse_wheel_callbacks=[<function dims_scroll at 0x0000022CF2A9EE80>], mouse_drag_callbacks=[<function drag_to_zoom at 0x0000022CF2A9EFC0>], mouse_double_click_callbacks=[<function double_click_to_zoom at 0x0000022CF2A9EF20>], camera=Camera(center=(0.0, 2600.0, 2525.0), zoom=0.17352432224572198, angles=(0.0, 0.0, 0.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(1.0, 1.0), viewbox=None, scaled=True, size=1.0, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=2, ndisplay=2, order=(0, 1), axis_labels=('-2', '-1'), rollable=(True, True), range=(RangeTuple(start=0.0, stop=5200.0, step=1.0), RangeTuple(start=0.0, stop=5050.0, step=1.0)), margin_left=(0.0, 0.0), margin_right=(0.0, 0.0), point=(2600.0, 2525.0), units=(<Unit('pixel')>, <Unit('pixel')>), last_used=0), grid=GridCanvas(str

In [193]:
qc_df['flag'].value_counts()

flag
OK                                                        20
Name: count, dtype: int64

In [194]:
list_channel_CLAHE=list(qc_df[qc_df['flag'].str.contains('CLAHE')]['biomarker'])

In [130]:
exclude = ('dna (', 'control', 'autofluorescence')
list_channel_CLAHE=[i for i in list_channel_CLAHE if not any(term in i for term in exclude)]
list_channel_CLAHE

['Ki67',
 'CD68',
 'CD8',
 'FOXP3',
 'ER',
 'CD57',
 'HLA-DRA',
 'HLA-DPB1',
 'CD15',
 'CD44',
 'dna']

In [ ]:
exclude = ('dna (', 'control', 'autofluorescence')
list_channel_ok = [i for i in list_channel_ok 
                   if not any(term in i for term in exclude)]


In [ ]:
channel=['dna','CD45']
image_test.find_biomarkers_of_interest(channel)

viewer = image_test.view_images(biomarkers_of_interest=list_channel_CLAHE, resolution_level=0)


Processing File: LSP12021-A7.ome.tif


{'dna': 0, 'CD45': 11}

# QC Manually

In [185]:
qc_dir = os.path.join(main_dir, 'QC')
os.makedirs(qc_dir, exist_ok=True)
print(qc_dir) 

D:\thu\HTAN\images\ILC\level_2\QC


In [ ]:
%gui qt

biomarkers_cyto_membrane = ['CD45','CD44']
flagged = qc_df[qc_df['flag'] != 'OK']
df_test_cyto_membrane=qc_df[qc_df['biomarker'].isin(biomarkers_cyto_membrane)]
viewer  = napari.Viewer()
widget  = qc_widget.QCWidget(viewer, df_test_cyto_membrane, 
                   save_path=os.path.join(qc_dir,f'{image_test.assessor.loader.file_name}_qc_decisions.csv'),
                   assessor=image_test.assessor,
                   resolution_level=0)
viewer.window.add_dock_widget(widget, name='QC Review')


In [ ]:
'GranzymeB', 'CD68', 'CD11c', 'CD11b', 'HLA-DRA', 'HLA-DPB1', 'CD44', 'HER2', 'PDL1-E1L3N', 'PDL1-28-8', 'PD1', 'CD45', 'CD3', 'CD4', 'CD8', 'CD20',

Viewer(mouse_move_callbacks=[], mouse_wheel_callbacks=[<function dims_scroll at 0x000001A64F39C860>], mouse_drag_callbacks=[<function drag_to_zoom at 0x000001A64F39C9A0>], mouse_double_click_callbacks=[<function double_click_to_zoom at 0x000001A64F39C900>], camera=Camera(center=(0.0, 1362.5, 1419.5), zoom=0.4554842259721203, angles=(0.0, 0.0, 0.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(1.0, 1.0), viewbox=None, scaled=True, size=1.0, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=2, ndisplay=2, order=(0, 1), axis_labels=('-2', '-1'), rollable=(True, True), range=(RangeTuple(start=0.0, stop=2725.0, step=1.0), RangeTuple(start=0.0, stop=2839.0, step=1.0)), margin_left=(0.0, 0.0), margin_right=(0.0, 0.0), point=(1362.0, 1419.0), units=(<Unit('pixel')>, <Unit('pixel')>), last_used=0), grid=GridCanvas(stri